In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

REPO_ROOT = Path("..").resolve()
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

jobs_clean = pd.read_parquet(DATA_PROCESSED / "jobs_clean.parquet")
resumes_clean = pd.read_parquet(DATA_PROCESSED / "resumes_clean.parquet")

job_emb = np.load(DATA_PROCESSED / "job_emb.npy")
resume_emb = np.load(DATA_PROCESSED / "resume_emb.npy")

print("jobs_clean:", jobs_clean.shape)
print("resumes_clean:", resumes_clean.shape)
print("job_emb:", job_emb.shape)
print("resume_emb:", resume_emb.shape)

jobs_clean: (1068, 6)
resumes_clean: (1200, 7)
job_emb: (1068, 384)
resume_emb: (1200, 384)


In [2]:
# -----------------------------
# Parse job experience ranges
# Example:
# '0-1' -> (0, 1)
# '10+ years' -> (10, None)
# -----------------------------
def parse_years_range(x: str):
    if x is None:
        return (None, None)

    s = str(x).lower().strip()
    s = s.replace("years", "").replace("year", "").strip()
    s = s.replace("–", "-")  # replace en-dash with normal hyphen

    if not s:
        return (None, None)

    # Pattern like "10+"
    m = re.match(r"(\d+)\s*\+", s)
    if m:
        return (int(m.group(1)), None)

    # Pattern like "4-7"
    m = re.match(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        return (int(m.group(1)), int(m.group(2)))

    # Pattern like "3"
    m = re.match(r"(\d+)", s)
    if m:
        v = int(m.group(1))
        return (v, v)

    return (None, None)


# -----------------------------
# Soft experience penalty
# Purpose:
# - if a resume has less experience than a job requires,
#   reduce the score without completely removing the job
# -----------------------------
def experience_penalty(resume_years: int, job_min):
    if job_min is None:
        return 1.0

    if resume_years < job_min:
        gap = job_min - resume_years
        # Never reduce below 0.4
        return max(0.4, 1.0 - 0.12 * gap)

    return 1.0


# -----------------------------
# Simple exact-match skill gap
# Used here for qualitative comparison only
# -----------------------------
def skill_gap(resume_skills, job_skills):
    # Convert both inputs into sets
    r = set(list(resume_skills) if resume_skills is not None else [])
    j = set(list(job_skills) if job_skills is not None else [])

    matched = sorted(r.intersection(j))
    missing = sorted(j.difference(r))

    return matched, missing

In [3]:
# -----------------------------
# Add numeric min/max experience columns to the jobs table
# This allows the enhanced ranker to apply the experience penalty
# -----------------------------
jobs_clean["min_years"], jobs_clean["max_years"] = zip(
    *jobs_clean["years_of_experience"].map(parse_years_range)
)

# Preview the parsed values
display(
    jobs_clean[["job_id", "job_title", "years_of_experience", "min_years", "max_years"]].head(10)
)

,job_id,job_title,years_of_experience,min_years,max_years
0,NET-F-001,.NET Developer,0-1,0,1.0
1,NET-F-002,.NET Developer,0-1,0,1.0
2,NET-F-003,.NET Developer,0-1,0,1.0
3,NET-F-004,.NET Developer,0-1,0,1.0
4,NET-F-005,.NET Developer,0-1,0,1.0
5,NET-F-006,.NET Developer,0-1,0,1.0
6,NET-F-007,.NET Developer,0-1,0,1.0
7,NET-F-008,.NET Developer,0-1,0,1.0
8,NET-F-009,.NET Developer,0-1,0,1.0
9,NET-F-010,.NET Developer,0-1,0,1.0


In [4]:
# -----------------------------
# Baseline ranking
# - uses cosine similarity only
# -----------------------------
def rank_jobs_baseline(resume_idx: int):
    # Compute similarity between one resume vector and all job vectors
    sims = cosine_similarity(
        resume_emb[resume_idx:resume_idx+1],
        job_emb
    )[0]

    # Return both raw scores and descending ranked indices
    return sims, np.argsort(sims)[::-1]


# -----------------------------
# Enhanced ranking
# - uses the final tuned formula:
#   final = 0.8 * semantic + 0.2 * (semantic * experience_penalty)
# -----------------------------
def rank_jobs_enhanced(resume_idx: int, w_sem=0.8, w_exp=0.2):
    # Semantic similarity scores
    sims = cosine_similarity(
        resume_emb[resume_idx:resume_idx+1],
        job_emb
    )[0]

    # Resume experience
    resume_years = int(resumes_clean.loc[resume_idx, "experience_years"])

    # Compute penalty for every job
    penalties = np.array([
        experience_penalty(resume_years, mn)
        for mn in jobs_clean["min_years"]
    ])

    # Final tuned score
    final = (w_sem * sims) + (w_exp * (sims * penalties))

    # Return semantic scores, final scores, penalties, and final ranking
    return sims, final, penalties, np.argsort(final)[::-1]

In [5]:
# -----------------------------
# Compare top-k recommendations for a single resume
# under both baseline and enhanced models
# -----------------------------
def compare_baseline_vs_enhanced(resume_id: str, k: int = 5):
    # Locate the resume row index
    r_idx = resumes_clean.index[resumes_clean["resume_id"] == resume_id][0]

    # Print basic resume information
    print("Resume ID:", resume_id)
    print("Current title:", resumes_clean.loc[r_idx, "current_job_title"])
    print("Experience years:", resumes_clean.loc[r_idx, "experience_years"])
    print("Resume skills:", resumes_clean.loc[r_idx, "resume_skills_list"])
    print("Target description:", resumes_clean.loc[r_idx, "target_job_description"])
    print("-" * 100)

    # -------------------------
    # Baseline results
    # -------------------------
    baseline_sims, baseline_ranked = rank_jobs_baseline(r_idx)
    baseline_top = baseline_ranked[:k]

    baseline_df = jobs_clean.iloc[baseline_top][
        ["job_id", "job_title", "experience_level", "years_of_experience", "job_skills_list"]
    ].copy()
    baseline_df["semantic_score"] = baseline_sims[baseline_top]

    # -------------------------
    # Enhanced results
    # -------------------------
    sims, final, penalties, enhanced_ranked = rank_jobs_enhanced(r_idx, 0.8, 0.2)
    enhanced_top = enhanced_ranked[:k]

    enhanced_df = jobs_clean.iloc[enhanced_top][
        ["job_id", "job_title", "experience_level", "years_of_experience", "job_skills_list"]
    ].copy()
    enhanced_df["semantic_score"] = sims[enhanced_top]
    enhanced_df["final_score"] = final[enhanced_top]
    enhanced_df["penalty"] = penalties[enhanced_top]

    # -------------------------
    # Add simple skill explanation
    # -------------------------
    resume_skills = resumes_clean.loc[r_idx, "resume_skills_list"]

    baseline_df["matched_skills"] = baseline_df["job_skills_list"].apply(
        lambda js: skill_gap(resume_skills, js)[0]
    )
    baseline_df["missing_skills"] = baseline_df["job_skills_list"].apply(
        lambda js: skill_gap(resume_skills, js)[1][:10]
    )

    enhanced_df["matched_skills"] = enhanced_df["job_skills_list"].apply(
        lambda js: skill_gap(resume_skills, js)[0]
    )
    enhanced_df["missing_skills"] = enhanced_df["job_skills_list"].apply(
        lambda js: skill_gap(resume_skills, js)[1][:10]
    )

    # -------------------------
    # Display results
    # -------------------------
    print("\nBASELINE TOP RESULTS")
    display(baseline_df.reset_index(drop=True))

    print("\nENHANCED TOP RESULTS")
    display(enhanced_df.reset_index(drop=True))

    return baseline_df.reset_index(drop=True), enhanced_df.reset_index(drop=True)

In [6]:
# -----------------------------
# Filter freshers only
# These are especially important because your platform is aimed at them
# -----------------------------
freshers = resumes_clean[resumes_clean["experience_years"] == 0].copy()
print("Freshers count:", len(freshers))

Freshers count: 441


In [7]:
# -----------------------------
# Find cases where the baseline top recommendation
# is clearly too senior / experienced for a fresher
# These are strong qualitative examples for the report
# -----------------------------
def find_bad_baseline_fresher_cases(limit=20):
    rows = []

    for r_idx in freshers.index:
        resume_id = resumes_clean.loc[r_idx, "resume_id"]
        current_title = resumes_clean.loc[r_idx, "current_job_title"]
        target_desc = resumes_clean.loc[r_idx, "target_job_description"]

        # Baseline ranking
        baseline_sims, baseline_ranked = rank_jobs_baseline(r_idx)
        top_job_idx = baseline_ranked[0]

        top_job_title = jobs_clean.loc[top_job_idx, "job_title"]
        top_job_level = jobs_clean.loc[top_job_idx, "experience_level"]
        top_job_years = jobs_clean.loc[top_job_idx, "years_of_experience"]

        # Heuristic: flag if top role is experienced/senior or requires 3+ years
        if (
            ("experienced" in str(top_job_level).lower())
            or ("senior" in str(top_job_level).lower())
            or ("lead" in str(top_job_level).lower())
            or (
                jobs_clean.loc[top_job_idx, "min_years"] is not None
                and jobs_clean.loc[top_job_idx, "min_years"] >= 3
            )
        ):
            rows.append({
                "resume_id": resume_id,
                "current_job_title": current_title,
                "experience_years": resumes_clean.loc[r_idx, "experience_years"],
                "target_job_description": target_desc,
                "baseline_top_job": top_job_title,
                "baseline_top_level": top_job_level,
                "baseline_top_years": top_job_years,
                "baseline_score": baseline_sims[top_job_idx]
            })

    return pd.DataFrame(rows).head(limit)

bad_cases = find_bad_baseline_fresher_cases(limit=20)
display(bad_cases)

,resume_id,current_job_title,experience_years,target_job_description,baseline_top_job,baseline_top_level,baseline_top_years,baseline_score
0,R_0000,,0,Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,Software Developer - Experienced,experienced,10+ years,0.709601
1,R_0013,,0,Looking for an opportunity as a Blockchain Engineer to leverage my expertise in software engineering and make a meaningful impact in the industry.,Blockchain Developer,experienced,5+,0.795225
2,R_0015,,0,Seeking a challenging role as a Quantum Computing Specialist where I can apply my skills and knowledge to contribute to organizational success and professional growth.,Blockchain Developer,experienced,3+,0.597455
3,R_0020,,0,Seeking a challenging role as a Mobile Applications Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,Cloud Engineer - Experienced,senior-level,7+ years,0.685314
4,R_0023,,0,Looking for an opportunity as a Prompt Engineer to leverage my expertise in cybersecurity and make a meaningful impact in the industry.,Cybersecurity Analyst,experienced,3-5,0.739166
5,R_0028,,0,"Looking for a Cloud Engineer role where I can architect cloud solutions, automate deployments, and optimize cloud resources for performance and cost-efficiency.",Cloud Engineer - Experienced,senior-level,6+ years,0.835667
6,R_0033,,0,Seeking a challenging role as a AI Ethics Officer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,AI Engineer - Experienced,senior-level,5+ years,0.720641
7,R_0036,,0,Looking for an opportunity as a Backend Developer to leverage my expertise in information technology and make a meaningful impact in the industry.,Blockchain Developer,experienced,6+,0.688473
8,R_0038,,0,Seeking a challenging role as a Blockchain Engineer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,Blockchain Developer,experienced,5+,0.813955
9,R_0039,,0,Looking for an opportunity as a Quantum Computing Specialist to leverage my expertise in artificial intelligence and make a meaningful impact in the industry.,AI Engineer - Experienced,senior-level,5+ years,0.599490


In [13]:
# -----------------------------
# Replace "R_0000" with any resume_id from bad_cases
# Then inspect how the enhanced model changes the ranking
# -----------------------------
compare_baseline_vs_enhanced("R_0028", k=5)

Resume ID: R_0028
Current title: 
Experience years: 0
Resume skills: ['spark' 'aws' 'jenkins' 'javascript' 'scrum' 'blockchain']
Target description: Looking for a Cloud Engineer role where I can architect cloud solutions, automate deployments, and optimize cloud resources for performance and cost-efficiency.
----------------------------------------------------------------------------------------------------

BASELINE TOP RESULTS


,job_id,job_title,experience_level,years_of_experience,job_skills_list,semantic_score,matched_skills,missing_skills
0,CL013,Cloud Engineer - Experienced,senior-level,6+ years,"[aws advanced, azure advanced, gcp advanced, terraform, cloudformation, ansible, docker, kubernetes, helm, ci, cd: jenkins, gitlab ci, circleci, cloud security, monitoring: cloudwatch, prometheus, elk, networking, storage, python, bash, powershell]",0.835667,[],"[ansible, aws advanced, azure advanced, bash, cd: jenkins, ci, circleci, cloud security, cloudformation, docker]"
1,CL020,Cloud Engineer - Experienced,senior-level,7+ years,"[aws advanced, azure advanced, gcp advanced, terraform, cloudformation, ansible, kubernetes, helm, docker swarm, ci, cd, cloud security, monitoring, networking, storage, python, bash, powershell]",0.829995,[],"[ansible, aws advanced, azure advanced, bash, cd, ci, cloud security, cloudformation, docker swarm, gcp advanced]"
2,CL016,Cloud Engineer - Experienced,senior-level,7+ years,"[aws advanced, azure advanced, gcp advanced, terraform, cloudformation, ansible, kubernetes, helm, docker swarm, ci, cd: jenkins, gitlab ci, circleci, cloud security, monitoring, logging: cloudwatch, prometheus, elk, networking, storage, python, bash, powershell]",0.825484,[],"[ansible, aws advanced, azure advanced, bash, cd: jenkins, ci, circleci, cloud security, cloudformation, docker swarm]"
3,CL019,Cloud Engineer - Experienced,senior-level,6+ years,"[aws advanced, azure advanced, gcp advanced, terraform, cloudformation, ansible, kubernetes, helm, docker swarm, ci, cd, cloud security, monitoring, networking, storage, python, bash, powershell]",0.824779,[],"[ansible, aws advanced, azure advanced, bash, cd, ci, cloud security, cloudformation, docker swarm, gcp advanced]"
4,CL014,Cloud Engineer - Experienced,senior-level,5–9 years,"[aws, azure, gcp, terraform, cloudformation, ansible, kubernetes, helm, docker swarm, ci, cd: jenkins, gitlab ci, circleci, cloud security, monitoring: cloudwatch, prometheus, elk, networking, storage, scripting: python, bash, powershell]",0.816022,[aws],"[ansible, azure, bash, cd: jenkins, ci, circleci, cloud security, cloudformation, docker swarm, elk]"



ENHANCED TOP RESULTS


,job_id,job_title,experience_level,years_of_experience,job_skills_list,semantic_score,final_score,penalty,matched_skills,missing_skills
0,CL003,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash, jenkins, aws codepipeline, monitoring: cloudwatch, azure monitor, git, github]",0.815228,0.815228,1.0,"[aws, jenkins]","[aws codepipeline, azure, azure monitor, bash, cloudformation, docker, gcp, git, github, kubernetes]"
1,CL007,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation knowledge, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring, git]",0.812869,0.812869,1.0,"[aws, jenkins]","[azure, bash scripting, cloudformation knowledge, docker, gcp, git, github actions, kubernetes, monitoring, python]"
2,CL008,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring: cloudwatch, prometheus, git, github]",0.805547,0.805547,1.0,"[aws, jenkins]","[azure, bash scripting, cloudformation, docker, gcp, git, github, github actions, kubernetes, monitoring: cloudwatch]"
3,CL004,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation awareness, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring, git]",0.801101,0.801101,1.0,"[aws, jenkins]","[azure, bash scripting, cloudformation awareness, docker, gcp, git, github actions, kubernetes, monitoring, python]"
4,CL005,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash scripting, jenkins, aws codepipeline, monitoring: cloudwatch, prometheus, git]",0.796774,0.796774,1.0,"[aws, jenkins]","[aws codepipeline, azure, bash scripting, cloudformation, docker, gcp, git, kubernetes, monitoring: cloudwatch, prometheus]"


(  job_id                     job_title experience_level years_of_experience  \
 0  CL013  Cloud Engineer - Experienced     senior-level            6+ years   
 1  CL020  Cloud Engineer - Experienced     senior-level            7+ years   
 2  CL016  Cloud Engineer - Experienced     senior-level            7+ years   
 3  CL019  Cloud Engineer - Experienced     senior-level            6+ years   
 4  CL014  Cloud Engineer - Experienced     senior-level           5–9 years   
 
                                                                                                                                                                                                                                                            job_skills_list  \
 0                 [aws advanced, azure advanced, gcp advanced, terraform, cloudformation, ansible, docker, kubernetes, helm, ci, cd: jenkins, gitlab ci, circleci, cloud security, monitoring: cloudwatch, prometheus, elk, networking, storage, pytho

In [9]:
# -----------------------------
# Find cases where the enhanced model changes the top result
# These are useful because they show the practical impact of reranking
# -----------------------------
def find_changed_top_results(limit=20):
    rows = []

    for r_idx in resumes_clean.index:
        resume_id = resumes_clean.loc[r_idx, "resume_id"]

        baseline_sims, baseline_ranked = rank_jobs_baseline(r_idx)
        _, final, penalties, enhanced_ranked = rank_jobs_enhanced(r_idx, 0.8, 0.2)

        base_top = baseline_ranked[0]
        enh_top = enhanced_ranked[0]

        if base_top != enh_top:
            rows.append({
                "resume_id": resume_id,
                "experience_years": resumes_clean.loc[r_idx, "experience_years"],
                "current_job_title": resumes_clean.loc[r_idx, "current_job_title"],
                "baseline_top_job": jobs_clean.loc[base_top, "job_title"],
                "baseline_top_level": jobs_clean.loc[base_top, "experience_level"],
                "baseline_semantic_score": baseline_sims[base_top],
                "enhanced_top_job": jobs_clean.loc[enh_top, "job_title"],
                "enhanced_top_level": jobs_clean.loc[enh_top, "experience_level"],
                "enhanced_final_score": final[enh_top],
                "enhanced_penalty": penalties[enh_top]
            })

    return pd.DataFrame(rows).head(limit)

changed_cases = find_changed_top_results(limit=20)
display(changed_cases)

,resume_id,experience_years,current_job_title,baseline_top_job,baseline_top_level,baseline_semantic_score,enhanced_top_job,enhanced_top_level,enhanced_final_score,enhanced_penalty
0,R_0000,0,,Software Developer - Experienced,experienced,0.709601,Cybersecurity Trainee,fresher,0.678482,1.00
1,R_0002,2,Prompt Engineer,AI Prompt Engineer,experienced,0.742856,AI Prompt Engineer,fresher,0.726339,1.00
2,R_0003,2,AI Engineer,AI Engineer - Experienced,senior-level,0.841704,AI Engineer - Fresher,entry-level,0.817829,1.00
3,R_0008,1,Data Scientist,Data Scientist - Experienced,mid-senior,0.737013,Data Engineer,fresher,0.720702,1.00
4,R_0011,5,Product Manager,Product Manager,experienced,0.699110,Product Manager,experienced,0.694806,1.00
5,R_0013,0,,Blockchain Developer,experienced,0.795225,Blockchain Developer,fresher,0.773500,1.00
6,R_0014,2,Blockchain Engineer,Blockchain Developer,experienced,0.786323,Blockchain Developer,fresher,0.780676,1.00
7,R_0015,0,,Blockchain Developer,experienced,0.597455,Fintech Engineer,fresher,0.569843,1.00
8,R_0016,1,Machine Learning Engineer,AI Engineer - Experienced,senior-level,0.774831,AI Engineer - Fresher,entry-level,0.753185,1.00
9,R_0017,3,Product Manager,AI Engineer - Experienced,senior-level,0.692909,Product Manager,experienced,0.678295,1.00


In [10]:
# -----------------------------
# Rank jobs using only the number of matched skills
# This is useful to show why pure skill overlap can be misleading
# -----------------------------
def top_skill_overlap_jobs_for_resume(resume_id: str, k: int = 10):
    # Find the resume
    r_idx = resumes_clean.index[resumes_clean["resume_id"] == resume_id][0]
    resume_skills = resumes_clean.loc[r_idx, "resume_skills_list"]

    rows = []

    for j_idx in jobs_clean.index:
        job_skills = jobs_clean.loc[j_idx, "job_skills_list"]
        matched, missing = skill_gap(resume_skills, job_skills)

        rows.append({
            "job_id": jobs_clean.loc[j_idx, "job_id"],
            "job_title": jobs_clean.loc[j_idx, "job_title"],
            "experience_level": jobs_clean.loc[j_idx, "experience_level"],
            "years_of_experience": jobs_clean.loc[j_idx, "years_of_experience"],
            "skill_overlap_count": len(matched),
            "matched_skills": matched
        })

    df = pd.DataFrame(rows).sort_values("skill_overlap_count", ascending=False).head(k)
    return df.reset_index(drop=True)

# Example usage
skill_only_df = top_skill_overlap_jobs_for_resume("R_0000", k=10)
display(skill_only_df)

,job_id,job_title,experience_level,years_of_experience,skill_overlap_count,matched_skills
0,WEB-E-011,Web Developer,experienced,3+,3,"[javascript, node.js, sql]"
1,WEB-E-020,Web Developer,experienced,5+,3,"[javascript, node.js, sql]"
2,WEB-E-012,Web Developer,experienced,4+,3,"[javascript, node.js, sql]"
3,8,Full Stack Developer - Entry Level,fresher,0–1 year,3,"[javascript, node.js, sql]"
4,WEB-E-014,Web Developer,experienced,3+,3,"[javascript, node.js, sql]"
5,13,Full Stack Developer - Experienced,experienced,3–5 years,3,"[javascript, node.js, sql]"
6,WEB-E-015,Web Developer,experienced,4+,3,"[javascript, node.js, sql]"
7,WEB-E-016,Web Developer,experienced,5+,3,"[javascript, node.js, sql]"
8,WEB-E-017,Web Developer,experienced,4+,3,"[javascript, node.js, sql]"
9,WEB-E-018,Web Developer,experienced,5+,3,"[javascript, node.js, sql]"


In [11]:
# -----------------------------
# Compare:
# - enhanced semantic ranking
# - naive skill-overlap ranking
# This helps support the argument that skill overlap alone is not enough
# -----------------------------
def compare_semantic_vs_skill_overlap(resume_id: str, k: int = 5):
    # Locate resume
    r_idx = resumes_clean.index[resumes_clean["resume_id"] == resume_id][0]

    # Enhanced semantic ranking
    sims, final, penalties, enhanced_ranked = rank_jobs_enhanced(r_idx, 0.8, 0.2)
    top_semantic = enhanced_ranked[:k]

    semantic_df = jobs_clean.iloc[top_semantic][
        ["job_id", "job_title", "experience_level", "years_of_experience", "job_skills_list"]
    ].copy()
    semantic_df["semantic_score"] = sims[top_semantic]
    semantic_df["final_score"] = final[top_semantic]

    # Skill-only ranking
    skill_df = top_skill_overlap_jobs_for_resume(resume_id, k=k)

    # Print resume context
    print("Resume ID:", resume_id)
    print("Resume skills:", resumes_clean.loc[r_idx, "resume_skills_list"])
    print("Target description:", resumes_clean.loc[r_idx, "target_job_description"])

    # Show semantic ranking
    print("\nENHANCED SEMANTIC RANKING")
    display(semantic_df.reset_index(drop=True))

    # Show skill-only ranking
    print("\nSKILL-OVERLAP RANKING ONLY")
    display(skill_df.reset_index(drop=True))

In [15]:
compare_semantic_vs_skill_overlap("R_0000", k=5)

Resume ID: R_0000
Resume skills: ['node.js' 'javascript' 'deep learning' 'statistics' 'sql']
Target description: Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.

ENHANCED SEMANTIC RANKING


,job_id,job_title,experience_level,years_of_experience,job_skills_list,semantic_score,final_score
0,CSA-F-009,Cybersecurity Trainee,fresher,0,"[c++, wireshark, metasploit, linux security, encryption]",0.678482,0.678482
1,CSA-F-008,Graduate Security Analyst,fresher,0,"[python, windows, linux, nist, snort, antivirus]",0.678026,0.678026
2,CSA-F-003,Cybersecurity Intern,fresher,0,"[python, linux, nist, snort, antivirus tools, vulnerability assessment]",0.672831,0.672831
3,CA004,Cloud Trainee,fresher,0,"[cloud storage, iam, python scripting, bash, basic networking]",0.671837,0.671837
4,CSA-F-001,Cybersecurity Analyst,fresher,0-1,"[python, c++, linux, firewall configuration, wireshark, nessus, snort, encryption, incident response]",0.661922,0.661922



SKILL-OVERLAP RANKING ONLY


,job_id,job_title,experience_level,years_of_experience,skill_overlap_count,matched_skills
0,WEB-E-011,Web Developer,experienced,3+,3,"[javascript, node.js, sql]"
1,WEB-E-020,Web Developer,experienced,5+,3,"[javascript, node.js, sql]"
2,WEB-E-012,Web Developer,experienced,4+,3,"[javascript, node.js, sql]"
3,8,Full Stack Developer - Entry Level,fresher,0–1 year,3,"[javascript, node.js, sql]"
4,WEB-E-014,Web Developer,experienced,3+,3,"[javascript, node.js, sql]"


In [16]:
compare_semantic_vs_skill_overlap("R_0028", k=5)

Resume ID: R_0028
Resume skills: ['spark' 'aws' 'jenkins' 'javascript' 'scrum' 'blockchain']
Target description: Looking for a Cloud Engineer role where I can architect cloud solutions, automate deployments, and optimize cloud resources for performance and cost-efficiency.

ENHANCED SEMANTIC RANKING


,job_id,job_title,experience_level,years_of_experience,job_skills_list,semantic_score,final_score
0,CL003,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash, jenkins, aws codepipeline, monitoring: cloudwatch, azure monitor, git, github]",0.815228,0.815228
1,CL007,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation knowledge, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring, git]",0.812869,0.812869
2,CL008,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring: cloudwatch, prometheus, git, github]",0.805547,0.805547
3,CL004,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation awareness, docker, kubernetes, python, bash scripting, jenkins, github actions, monitoring, git]",0.801101,0.801101
4,CL005,Cloud Engineer - Fresher,entry-level,0–1 year,"[aws, azure, gcp, terraform, cloudformation, docker, kubernetes, python, bash scripting, jenkins, aws codepipeline, monitoring: cloudwatch, prometheus, git]",0.796774,0.796774



SKILL-OVERLAP RANKING ONLY


,job_id,job_title,experience_level,years_of_experience,skill_overlap_count,matched_skills
0,FE-F-010,Fintech Engineer,fresher,0-1,3,"[aws, blockchain, javascript]"
1,FE-F-007,Fintech Engineer,fresher,0-1,3,"[aws, blockchain, javascript]"
2,FE-F-006,Fintech Engineer,fresher,0-1,3,"[aws, blockchain, javascript]"
3,FE-F-008,Fintech Engineer,fresher,0-1,3,"[aws, blockchain, javascript]"
4,FE-F-009,Fintech Engineer,fresher,0-1,3,"[aws, blockchain, javascript]"



  ## Qualitative Case Study Summary

This notebook was used to perform qualitative analysis of the recommendation system in order to complement the quantitative evaluation results. The aim was to identify concrete examples showing how the enhanced ranking model behaves differently from the baseline model and why those differences matter in practice.

The analysis focused on three main questions:

1. **Does the enhanced model correct unrealistic baseline recommendations?**  
   In particular, fresher resumes were examined to see whether the baseline model recommended overly senior or experienced roles, and whether the experience-aware reranking shifted these recommendations toward more realistic entry-level or trainee positions.

2. **Does the enhanced model preserve semantic relevance while improving realism?**  
   The comparison checked whether the reranking step kept the same general role family (for example, cloud engineering or cybersecurity) while adjusting the level of seniority.

3. **Would simple skill-overlap ranking have been a better alternative?**  
   Semantic ranking was compared against naive skill-overlap ranking in order to test whether literal skill matching would have produced more useful results. This helped justify the decision to keep skills for explanation and gap analysis rather than as a primary ranking signal.

The qualitative examples identified in this notebook support the main findings of the evaluation phase:

- the baseline model can identify the correct job domain, but may return unrealistic senior roles for freshers
- the enhanced model improves recommendation realism by incorporating experience-aware reranking
- literal skill-overlap ranking can be misleading because it may prioritize surface-level technical overlap over broader role context

Overall, this notebook provides case-study evidence that the enhanced model adds practical value beyond the raw metric improvements reported in the quantitative evaluation.section